# **MONTE CARLO INTEGRATION:**
=============================================
- **DEFINITION:**
- Let $f: (0, 1)^d \to \mathbb{R}, f \in C^0((0, 1)^d)$. Consider a sequence of d-dimensional vector valued i.i.d. random variables $\{X_i\}_{i \in \mathbb{N}}$ having uniform distribution on $(0, 1)^d$. We call:
$$
\lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^{n} f(X_i)
$$

- the **Monte Carlo Approximation** of $\int_{[0, 1]^d} f(x)dx$ **MONTE CARLO INTEGRATION OF f** with n samples. 

**GENERAL FORM:**

For general domains one requires a suitable transformation.

For $d = 1$ and general $a, b \in \mathbb{R}$, the integral $\int_a^b f(u)\,du$ can be expressed by the transformation $u = a + x \cdot (b - a)$ with $x \in [0,1]$. We get:

$$
\int_a^b f(u)\,du = (b - a) \cdot \int_0^1 f(a + x \cdot (b - a))\,dx.
$$

For general dimensions $d$ we have, with an injective differentiable transformation  
$g : [0,1]^d \rightarrow \mathbb{R}^d$, that:

$$
\int_{g([0,1]^d)} f(u)\,du = \int_{[0,1]^d} f(g(x)) \cdot \left| \det\left( \frac{dg(x)}{dx} \right) \right|\,dx,
$$

Such a transformation may be combined with an indicator function on the domain.  
For $A \subset [0,1]^d$ we have:

$$
\int_A f(x)\,dx = \int_{[0,1]^d} f(x) \cdot \mathbf{1}_A(x)\,dx,
$$

- $\mathbf{1}_A(x)$ is the indicator function of the set A, i.e. $\mathbf{1}_A(x) = 1$ if $x \in A$ and $\mathbf{1}_A(x) = 0$ otherwise.


**`PROOF`**

In [1]:
import random
import numpy as np
import math 
import time
from concurrent.futures import ThreadPoolExecutor #Execute computations asynchronously using threads or processes.
from scipy.stats.qmc import Halton
import multiprocessing
from scipy.stats import qmc
from numpy.random import MT19937, Generator


In [2]:
class Integrator1D:
    def integrate(self, integrand, lower_bound, upper_bound):
        """
        Interface method to be overridden by subclasses.
        """
        raise NotImplementedError("Subclasses should implement this!")

### Monte Carlo Integration Formula in 1D

Let $f(x)$ be an integrable function on the interval $[a, b]$, and let $n$ be the number of evaluation points. Then, the integral can be approximated as:

$$
\int_a^b f(x)\, dx \approx (b - a) \cdot \frac{1}{n} \sum_{i=1}^{n} f(x_i)
$$

where:

- $x_i \sim \mathcal{U}(a, b)$, i.e., each $x_i$ is a uniform random sample in the interval $[a, b]$
- $n$ is the number of evaluation points (`number_of_evaluation_points`)
- $x_i = a + u_i \cdot (b - a)$, with $u_i \sim \mathcal{U}(0,1)$

---

### Algorithm Steps

1. Generate $n$ uniform samples $u_i \sim \mathcal{U}(0,1)$  
2. Transform them to the domain $[a, b]$:

$$
x_i = a + u_i \cdot (b - a)
$$

3. Evaluate the function:

$$
f_i = f(x_i)
$$

4. Compute the average:

$$
\bar{f} = \frac{1}{n} \sum_{i=1}^{n} f_i
$$

5. Multiply by the size of the domain:

$$
\int_a^b f(x)\, dx \approx (b - a) \cdot \bar{f}
$$


In [3]:
class MonteCarloIntegrator1D(Integrator1D):
    def __init__(self, number_of_evaluation_points, seed):
        self.number_of_evaluation_points = number_of_evaluation_points
        self.seed = seed

    def integrate(self, integrand, lower_bound, upper_bound):
        np.random.seed(self.seed)
        domain_size = upper_bound - lower_bound 

        sum = 0.0

        for i in range(self.number_of_evaluation_points):
            random_number = np.random.uniform(0.0, 1.0)
            argument = lower_bound + random_number*domain_size
            value = integrand(argument)

            sum += value

        return sum / self.number_of_evaluation_points*domain_size


In [4]:
class MonteCarloIntegrator1DWithStreams(Integrator1D):
    def __init__(self, number_of_evaluation_points, seed=3141):
        self.number_of_evaluation_points = number_of_evaluation_points
        self.seed = seed

    def integrate(self, integrand, lower_bound, upper_bound):
        rng = random.Random(self.seed)
        domain_size = upper_bound - lower_bound

        random_numbers = [rng.random() for _ in range(self.number_of_evaluation_points)]
        sum_values = sum(integrand(lower_bound + x * domain_size) for x in random_numbers)

        return sum_values / self.number_of_evaluation_points * domain_size

### Simpson’s Rule Integration Formula in 1D

Let $f(x)$ be a sufficiently smooth function on the interval $[a, b]$, and let $n$ be the number of evaluation points (an odd number). Then, the integral can be approximated using the composite Simpson’s rule:

$$
\int_a^b f(x)\, dx \approx \frac{h}{3} \left[ f(x_0) + 4 \sum_{\text{odd } i=1}^{n-2} f(x_i) + 2 \sum_{\text{even } i=2}^{n-3} f(x_i) + f(x_{n-1}) \right]
$$

where:

- $x_0 = a$, $x_{n-1} = b$
- $n$ is the number of evaluation points, and must be **odd**
- $h = \frac{b - a}{n - 1}$ is the spacing between consecutive points
- The $x_i$ are evenly spaced: $x_i = a + i \cdot h$

---

### Algorithm Steps

1. Compute the step size:

$$
h = \frac{b - a}{n - 1}
$$

2. Generate $n$ evenly spaced points:

$$
x_i = a + i \cdot h, \quad \text{for } i = 0, 1, \dots, n - 1
$$

3. Evaluate the function at all $x_i$:

$$
f_i = f(x_i)
$$

4. Apply Simpson's weights:

- Use weight 1 for $f(x_0)$ and $f(x_{n-1})$
- Use weight 4 for all **odd-indexed** interior points
- Use weight 2 for all **even-indexed** interior points (excluding ends)

5. Combine with:

$$
\int_a^b f(x)\, dx \approx \frac{h}{3} \left[ f_0 + 4 \sum_{\text{odd } i} f_i + 2 \sum_{\text{even } i \ne 0,n-1} f_i + f_{n-1} \right]
$$


In [5]:
class SimpsonsIntegrator1D(Integrator1D):
    def __init__(self, number_of_evaluation_points):
        if number_of_evaluation_points % 2 != 1: 
            raise ValueError("number_of_evaluation_points must be odd")
        self.number_of_evaluation_points = number_of_evaluation_points

    def integrate(self, integrand, lower_bound, upper_bound):
        n = self.number_of_evaluation_points
        h = (upper_bound - lower_bound) / (n - 1) #Step size

        integral = integrand(lower_bound) + integrand(upper_bound)

        for i in range(1, n - 1):
            x = lower_bound + i * h
            weight = 4 if i % 2 != 0 else 2
            integral += weight * integrand(x)

        return integral * h / 3

In [6]:
class SimpsonsIntegrator1DWithStreams(Integrator1D):
    def __init__(self, number_of_evaluation_points):
        self.number_of_evaluation_points = number_of_evaluation_points
        if number_of_evaluation_points % 2 != 1:
            raise ValueError("number_of_evaluation_points should be odd")

    def integrate(self, integrand, lower_bound, upper_bound):
        domain_size = upper_bound - lower_bound
        number_of_double_intervals = (self.number_of_evaluation_points - 1) // 2
        interval_size = domain_size / number_of_double_intervals / 2.0  # h

        # Suma principal con pesos 2 y 4 en los puntos internos
        sum_internal = sum(
            2 * integrand(lower_bound + 2 * i * interval_size) +
            4 * integrand(lower_bound + (2 * i + 1) * interval_size)
            for i in range(1, number_of_double_intervals)
        )

        # Agregamos los extremos y el primer punto impar
        sum_total = (
            sum_internal
            + 4 * integrand(lower_bound + interval_size)
            + integrand(lower_bound)
            + integrand(upper_bound)
        )

        return sum_total * interval_size / 3


In [ ]:
class integrator_1D_experiment():

    def test_integrator(self, integrator, name = ""):
        integrand = lambda x: math.cos(x)
        integral_analytic = lambda x: math.sin(x)
    
        lower_bound = 0.0
        upper_bound = 5.0

        integral_value_analytic = integrand(upper_bound) - integral_analytic(lower_bound)
        integral_value_integrator = integrator.integrate(integrand, lower_bound, upper_bound)
        error = integral_value_integrator - integral_value_analytic

        print(f"{name:<42}  {integral_value_integrator:20.16f}  ± {abs(error):5.3e}")
    
    def main(self):
        print("""Testing several implementations of a 1D integrator.
              (Note that some implementations may use Kahan summation or similar techniques.)""")
        
        number_of_evaluation_points = 10001
        print(f"Number of evaluation points....: {number_of_evaluation_points} (≈ {number_of_evaluation_points:.2e})\n")

        print("Theoretical (relative) errors are:")
        print(f"\tMonte-Carlo integration.......(1/n)^0.5..: {math.pow(1.0 / number_of_evaluation_points, 0.5):.2e}")
        print(f"\tSimpson's rule integration....(1/n)^4....: {math.pow(1.0 / number_of_evaluation_points, 4.0):.2e}\n")

        integrators = [
            (SimpsonsIntegrator1D(number_of_evaluation_points), "SimpsonsIntegrator1D"),
            (SimpsonsIntegrator1DWithStreams(number_of_evaluation_points), "SimpsonsIntegrator1DWithStreams"),
            (MonteCarloIntegrator1D(number_of_evaluation_points, 3141), "MonteCarloIntegrator1D"),
            (MonteCarloIntegrator1DWithStreams(number_of_evaluation_points, 3141), "MonteCarloIntegrator1DWithStreams"),
            (MonteCarloIntegrator1DWithStreams(number_of_evaluation_points, 3141), "MonteCarloIntegrator1DWithStreams"),
        ]

        for integrator, name in integrators:
            self.test_integrator(integrator, name)


In [8]:
a = integrator_1D_experiment()
a.main()

Testing several implementations of a 1D integrator.
              (Note that some implementations may use Kahan summation or similar techniques.)
Number of evaluation points....: 10001 (≈ 1.00e+04)

Theoretical (relative) errors are:
	Monte-Carlo integration.......(1/n)^0.5..: 1.00e-02
	Simpson's rule integration....(1/n)^4....: 1.00e-16

SimpsonsIntegrator1D                         -0.9589242746631377  ± 1.243e+00
SimpsonsIntegrator1DWithStreams              -0.9589242746631389  ± 1.243e+00
MonteCarloIntegrator1D                       -0.9921573721286792  ± 1.276e+00
MonteCarloIntegrator1DWithStreams            -0.9523324603356010  ± 1.236e+00


**`EXAMPLE: INTGRATION OF UNIT CIRCLE ($\pi$)`**

In [9]:
def get_van_der_corput_number(index: int, base: int) -> float:
    index += 1  # Como en Java: index = index + 1

    x = 0.0
    refinement_factor = 1.0 / base

    while index > 0:
        x += (index % base) * refinement_factor
        index //= base  # División entera
        refinement_factor /= base

    return x


> This experiment compares three methods for approximating the integral of the deterministic function $f(x) = x^3$ over $[0,1]$: pseudo-random sampling (Mersenne Twister), equidistant sampling, and quasi-random sampling (Van der Corput). The goal is to evaluate their accuracy in Monte Carlo integration.


In [10]:
class montecarlo_integration_experiment():
    def main():
        function = lambda x: x*x*x
        integral_analytic = 0.25
        number_of_sample_points = 100000

        mersenne = np.random.default_rng(seed=3141)

        print("Integration errors:")
        print("n\tmersenne\tequidistant\tv.-d.-corput")

        sum_mersenne_twister = 0.0
        sum_equidistributed = 0.0
        sum_van_der_corput = 0.0

        for i in range(number_of_sample_points):
            sum_mersenne_twister += function(mersenne.random())
            sum_equidistributed += function(i/number_of_sample_points)
            sum_van_der_corput += function(get_van_der_corput_number(i, 2))

            current_number_of_samples = i+2

            integral_mersenne_twister  = sum_mersenne_twister  / number_of_sample_points
            error_mersenne_twister  = integral_mersenne_twister  - integral_analytic

            integral_equidistributed = sum_equidistributed / number_of_sample_points
            error_equidistributed = integral_equidistributed - integral_analytic

            integral_van_der_corput = sum_van_der_corput / number_of_sample_points
            error_van_der_corput = integral_van_der_corput - integral_analytic

            #Print every 100 result
            if current_number_of_samples % 100 == 0:
                print(f"{current_number_of_samples}\t"
                            f"{error_mersenne_twister:.3E}\t"
                            f"{error_equidistributed:.3E}\t"
                            f"{error_van_der_corput:.3E}")

        #Calculate the final result
        integral_mersenne_twister  = sum_mersenne_twister  / number_of_sample_points
        error_mersenne_twister  = integral_mersenne_twister  - integral_analytic

        integral_equisdistributed = sum_equidistributed / number_of_sample_points
        error_equidistributed = integral_equidistributed - integral_analytic

        integral_van_der_corput = sum_van_der_corput / number_of_sample_points
        error_van_der_corput = integral_van_der_corput - integral_analytic

        print("\nFinal results:")
        print(f"Pseudo RNG....: {integral_mersenne_twister:.6f}\t error: {error_mersenne_twister :.6f}")
        print(f"Equidistri....: {integral_equidistributed:.6f}\t error: {error_equidistributed:.6f}")
        print(f"v.d.Corput....: {integral_van_der_corput:.6f}\t error: {error_van_der_corput :.6f}")

In [11]:
pi = montecarlo_integration_experiment
pi.main()

Integration errors:
n	mersenne	equidistant	v.-d.-corput
100	-2.498E-01	-2.500E-01	-2.498E-01
200	-2.495E-01	-2.500E-01	-2.495E-01
300	-2.493E-01	-2.500E-01	-2.493E-01
400	-2.490E-01	-2.500E-01	-2.490E-01
500	-2.488E-01	-2.500E-01	-2.488E-01
600	-2.485E-01	-2.500E-01	-2.485E-01
700	-2.483E-01	-2.500E-01	-2.483E-01
800	-2.480E-01	-2.500E-01	-2.480E-01
900	-2.478E-01	-2.500E-01	-2.478E-01
1000	-2.475E-01	-2.500E-01	-2.475E-01
1100	-2.472E-01	-2.500E-01	-2.473E-01
1200	-2.470E-01	-2.500E-01	-2.470E-01
1300	-2.468E-01	-2.500E-01	-2.468E-01
1400	-2.465E-01	-2.500E-01	-2.465E-01
1500	-2.463E-01	-2.500E-01	-2.463E-01
1600	-2.460E-01	-2.500E-01	-2.460E-01
1700	-2.457E-01	-2.500E-01	-2.458E-01
1800	-2.455E-01	-2.500E-01	-2.455E-01
1900	-2.452E-01	-2.500E-01	-2.453E-01
2000	-2.450E-01	-2.500E-01	-2.450E-01
2100	-2.447E-01	-2.500E-01	-2.448E-01
2200	-2.444E-01	-2.500E-01	-2.445E-01
2300	-2.442E-01	-2.500E-01	-2.443E-01
2400	-2.440E-01	-2.500E-01	-2.440E-01
2500	-2.437E-01	-2.500E-01	-2.438E-01
260

In [12]:
def get_monte_carlo_approx_pi(number_of_simulations):
    number_of_points_inside_UnCircle = 0
    
    for i in range(number_of_simulations):
        x = 2 * (np.random.random()-0.5)
        y = 2 * (np.random.random()-0.5)

        if (x*x + y*y) < 1.0:
            number_of_points_inside_UnCircle += 1

    area_of_unitCirc = 4*number_of_points_inside_UnCircle/number_of_simulations

    pi = area_of_unitCirc

    return pi

In [13]:
get_monte_carlo_approx_pi(1000)

3.072

In [ ]:

class MonteCarloIntegrationParallelExperiment:
    piAnalytic = math.pi

    @staticmethod
    def main():
        numberOfSamples = 200_000

        print("Monte-Carlo approximation of Pi:                                  error           time")
        print("_" * 100)

        MonteCarloIntegrationParallelExperiment.testHaltonWithStreamSeq(numberOfSamples)
        MonteCarloIntegrationParallelExperiment.testHaltonWithStreamPar(numberOfSamples)
        MonteCarloIntegrationParallelExperiment.testMersenneWithStreamSeq(numberOfSamples)
        MonteCarloIntegrationParallelExperiment.testMersenneWithStreamPar(numberOfSamples)
        MonteCarloIntegrationParallelExperiment.testHaltonWithExecutor(numberOfSamples)
        MonteCarloIntegrationParallelExperiment.testMersenneWithExecutor(numberOfSamples)

    @staticmethod
    def testHaltonWithStreamSeq(numberOfSamples):
        start = time.time()
        sampler = qmc.Halton(d=2, scramble=False)
        points = sampler.random(n=numberOfSamples)
        x, y = 2 * points[:, 0] - 1, 2 * points[:, 1] - 1
        inside = (x**2 + y**2) < 1
        pi_est = 4.0 * np.sum(inside) / numberOfSamples
        end = time.time()
        print(f"Halton, sequential using stream..............................: {pi_est - MonteCarloIntegrationParallelExperiment.piAnalytic:10.2E}\t{end - start:.2f} sec.")

    @staticmethod
    def testHaltonWithStreamPar(numberOfSamples):
        start = time.time()
        sampler = qmc.Halton(d=2, scramble=False)
        points = sampler.random(n=numberOfSamples)
        x, y = 2 * points[:, 0] - 1, 2 * points[:, 1] - 1
        inside = (x**2 + y**2) < 1
        pi_est = 4.0 * np.sum(inside) / numberOfSamples
        end = time.time()
        print(f"Halton, parallel using stream................................: {pi_est - MonteCarloIntegrationParallelExperiment.piAnalytic:10.2E}\t{end - start:.2f} sec.")

    @staticmethod
    def testMersenneWithStreamSeq(numberOfSamples):
        start = time.time()
        rng = Generator(MT19937(seed=3141))
        x = 2 * rng.random(numberOfSamples) - 1
        y = 2 * rng.random(numberOfSamples) - 1
        inside = (x**2 + y**2) < 1
        pi_est = 4.0 * np.sum(inside) / numberOfSamples
        end = time.time()
        print(f"Mersenne, sequential using stream............................: {pi_est - MonteCarloIntegrationParallelExperiment.piAnalytic:10.2E}\t{end - start:.2f} sec.")

    @staticmethod
    def testMersenneWithStreamPar(numberOfSamples):
        start = time.time()
        rng = Generator(MT19937(seed=3141))
        x = 2 * rng.random(numberOfSamples) - 1
        y = 2 * rng.random(numberOfSamples) - 1
        inside = (x**2 + y**2) < 1
        pi_est = 4.0 * np.sum(inside) / numberOfSamples
        end = time.time()
        print(f"Mersenne, parallel using stream, synchronized.................: {pi_est - MonteCarloIntegrationParallelExperiment.piAnalytic:10.2E}\t{end - start:.2f} sec.")

    @staticmethod
    def testMersenneWithExecutor(numberOfSamples):
        start = time.time()
        num_tasks = 100
        samples_per_task = numberOfSamples // num_tasks
        random_seed = random.Random(3216)

        def task(seed):
            return MonteCarloIntegrationParallelExperiment.getApproximationOfPiWithMersenne(seed, samples_per_task)

        with ThreadPoolExecutor() as executor:
            seeds = [random_seed.randint(0, 2**32 - 1) for _ in range(num_tasks)]
            results = list(executor.map(task, seeds))

        pi_est = sum(results) / num_tasks
        end = time.time()
        print(f"Mersenne, parallel Executor w/ thread local generator........: {pi_est - MonteCarloIntegrationParallelExperiment.piAnalytic:10.2E}\t{end - start:.2f} sec.")

    @staticmethod
    def testHaltonWithExecutor(numberOfSamples):
        start = time.time()
        num_tasks = 100
        samples_per_task = numberOfSamples // num_tasks

        def task(start_index):
            return MonteCarloIntegrationParallelExperiment.getApproximationOfPiWithHalton(start_index, samples_per_task)

        with ThreadPoolExecutor() as executor:
            indices = [i * samples_per_task for i in range(num_tasks)]
            results = list(executor.map(task, indices))

        pi_est = sum(results) / num_tasks
        end = time.time()
        print(f"Halton, parallel using Executor..............................: {pi_est - MonteCarloIntegrationParallelExperiment.piAnalytic:10.2E}\t{end - start:.2f} sec.")

    @staticmethod
    def getApproximationOfPiWithMersenne(seed, numberOfSamples):
        rng = Generator(MT19937(seed))
        x = 2 * rng.random(numberOfSamples) - 1
        y = 2 * rng.random(numberOfSamples) - 1
        inside = np.sum(x**2 + y**2 < 1)
        return 4.0 * inside / numberOfSamples

    @staticmethod
    def getApproximationOfPiWithHalton(startIndex, numberOfSamples):
        sampler = qmc.Halton(d=2, scramble=False)
        sampler.fast_forward(startIndex)
        points = sampler.random(n=numberOfSamples)
        x = 2 * points[:, 0] - 1
        y = 2 * points[:, 1] - 1
        inside = np.sum(x**2 + y**2 < 1)
        return 4.0 * inside / numberOfSamples


# Ejecutar el experimento
if __name__ == "__main__":
    MonteCarloIntegrationParallelExperiment.main()


Monte-Carlo approximation of Pi:                                  error           time
____________________________________________________________________________________________________
Halton, sequential using stream..............................:   1.03E-06	17.23 sec.
Halton, parallel using stream................................:   1.03E-06	17.96 sec.
Mersenne, sequential using stream............................:   8.13E-06	4.87 sec.
Mersenne, parallel using stream, synchronized.................:   8.13E-06	4.87 sec.
Halton, parallel using Executor..............................:   1.03E-06	6617.06 sec.
Mersenne, parallel Executor w/ thread local generator........:   7.74E-05	1.34 sec.
